# Emu3 Chat Template Application
Apply chat templates using the Llama3 Emu3 tokenizer

In [1]:
import json
from transformers import AutoTokenizer
from typing import List, Dict, Optional

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the Emu3 tokenizer
tokenizer_path = "/capstor/store/cscs/swissai/infra01/MLLM/llama3_vision_instruct_emu3_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    trust_remote_code=True,
    use_fast=True
)

print(f"Tokenizer loaded from: {tokenizer_path}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")

Tokenizer loaded from: /capstor/store/cscs/swissai/infra01/MLLM/llama3_vision_instruct_emu3_tokenizer
Vocab size: 128000
Model max length: 1000000000000000019884624838656


In [3]:
len(tokenizer)

161129

In [4]:
tokenizer.encode('<|start_header_id|>user<|end_header_id|>', add_special_tokens=False)

[128006, 882, 128007]

In [5]:
# Check available special tokens
print("Special tokens:")
print(f"BOS token: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"\nAdditional special tokens:")
for token in tokenizer.additional_special_tokens[:10]:  # Show first 10
    print(f"  {token}")

Special tokens:
BOS token: <|begin_of_text|> (ID: 128000)
EOS token: <|end_of_text|> (ID: 128001)
PAD token: None (ID: None)

Additional special tokens:
  <|img_start|>
  <|img_end|>
  <|img_token_start|>
  <|img_end_of_row|>
  <|img_end_of_frame|>
  <|RESERVED_000|>
  <|RESERVED_001|>
  <|image|>
  <|RESERVED_003|>
  <|RESERVED_004|>


In [6]:
# Check if chat template is available
if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template:
    print("Chat template found!")
    print(tokenizer.chat_template + "...")
else:
    print("No chat template found in tokenizer")

Chat template found!
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
    {%- set user_supplied_system_message = true %}
{%- else %}
    {%- set system_message = "" %}
    {%- set user_supplied_system_message = false %}
{%- endif %}

{#- Find out if there are any images #}
{% set image_ns = namespace(has_images=false

## Basic Chat Template Application

In [7]:
# Example 1: Conversation with image
# Using Llama-3.2-Vision-Instruct format with image placeholder
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},  # This becomes <|image|> token (ID 128263)
            {"type": "text", "text": "What do you see in this image? Describe it in detail."}
        ]
    },
    {
        "role": "assistant",
        "content": "Looking at the image, I can see..."
    }
]

# Apply chat template
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    add_special_tokens=False
)

print("Chat template output:")
print(chat_text)
print()

# Tokenize to see the token IDs
tokens = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True
)

print(f"Tokenized: {len(tokens)} tokens")
print(f"Token IDs: {tokens[:30]}...")
print()

# Check if image token is present
image_token_id = 128263  # <|image|> token ID
if image_token_id in tokens:
    position = tokens.index(image_token_id)
    print(f"✅ Image token (ID {image_token_id}) found at position {position}")
else:
    print(f"⚠️ Image token (ID {image_token_id}) not found - check tokenizer configuration")

Chat template output:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|image|>What do you see in this image? Describe it in detail.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Looking at the image, I can see...<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Tokenized: 38 tokens
Token IDs: [128000, 128006, 882, 128007, 271, 128263, 3923, 656, 499, 1518, 304, 420, 2217, 30, 61885, 433, 304, 7872, 13, 128009, 128006, 78191, 128007, 271, 23274, 520, 279, 2217, 11, 358]...

✅ Image token (ID 128263) found at position 5


In [25]:
tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    add_special_tokens=False
)

[128000,
 128006,
 882,
 128007,
 271,
 128263,
 3923,
 656,
 499,
 1518,
 304,
 420,
 2217,
 30,
 61885,
 433,
 304,
 7872,
 13,
 128009,
 128006,
 78191,
 128007,
 271,
 23274,
 520,
 279,
 2217,
 11,
 358,
 649,
 1518,
 1131,
 128009]

In [9]:
# Tokenize the chat
tokens = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

print(f"Token shape: {tokens.shape}")
print(f"Number of tokens: {tokens.shape[-1]}")
print(f"\nFirst 50 tokens: {tokens[0][:50].tolist()}")

Token shape: torch.Size([1, 38])
Number of tokens: 38

First 50 tokens: [128000, 128006, 882, 128007, 271, 128263, 3923, 656, 499, 1518, 304, 420, 2217, 30, 61885, 433, 304, 7872, 13, 128009, 128006, 78191, 128007, 271, 23274, 520, 279, 2217, 11, 358, 649, 1518, 1131, 128009, 128006, 78191, 128007, 271]


## Multi-turn Conversation

In [10]:
# Multi-turn conversation
multi_turn_messages = [
    {"role": "user", "content": "Can you help me understand Python lists?"},
    {"role": "assistant", "content": "Of course! Python lists are ordered, mutable collections that can store multiple items."},
    {"role": "user", "content": "How do I add items to a list?"},
    {"role": "assistant", "content": "You can use append() to add one item, extend() to add multiple items, or insert() to add at a specific position."},
    {"role": "user", "content": "Show me an example with append."}
]

# Apply template
multi_turn_text = tokenizer.apply_chat_template(
    multi_turn_messages,
    tokenize=False,
    add_generation_prompt=False,
    add_special_tokens=True,
)

print("Multi-turn conversation:")
print(multi_turn_text)

Multi-turn conversation:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Can you help me understand Python lists?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Of course! Python lists are ordered, mutable collections that can store multiple items.<|eot_id|><|start_header_id|>user<|end_header_id|>

How do I add items to a list?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

You can use append() to add one item, extend() to add multiple items, or insert() to add at a specific position.<|eot_id|><|start_header_id|>user<|end_header_id|>

Show me an example with append.<|eot_id|>


## Handling Image Tokens (Emu3 specific)

In [11]:
# Check for image-related special tokens
image_tokens = [token for token in tokenizer.additional_special_tokens if 'img' in token.lower()]
print(f"Image-related tokens found: {len(image_tokens)}")
for token in image_tokens[:5]:  # Show first 5
    token_id = tokenizer.convert_tokens_to_ids(token)
    print(f"  {token} -> ID: {token_id}")

Image-related tokens found: 5
  <|img_start|> -> ID: 128256
  <|img_end|> -> ID: 128257
  <|img_token_start|> -> ID: 128258
  <|img_end_of_row|> -> ID: 128259
  <|img_end_of_frame|> -> ID: 128260


In [12]:
# Example with image placeholders
# Emu3 uses special tokens for images
image_message = [
    {"role": "user", "content": "<|img_start|>[IMAGE_TOKENS]<|img_end|> What is in this image?"},
    {"role": "assistant", "content": "I can see..."}
]

# Apply template
image_chat = tokenizer.apply_chat_template(
    image_message,
    tokenize=False,
    add_generation_prompt=False
)

print("Chat with image placeholders:")
print(image_chat)

Chat with image placeholders:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|img_start|>[IMAGE_TOKENS]<|img_end|> What is in this image?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I can see...<|eot_id|>


## Batch Processing

In [13]:
# Process multiple conversations in batch
batch_conversations = [
    [
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "2+2 equals 4."}
    ],
    [
        {"role": "user", "content": "Tell me about cats."},
        {"role": "assistant", "content": "Cats are domestic felines."}
    ],
    [
        {"role": "user", "content": "How's the weather?"}
    ]
]

# Process each conversation
for i, conv in enumerate(batch_conversations):
    print(f"\nConversation {i+1}:")
    formatted = tokenizer.apply_chat_template(
        conv,
        tokenize=False,
        add_generation_prompt=True
    )
    print(formatted)
    print("-" * 40)


Conversation 1:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

What is 2+2?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

2+2 equals 4.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


----------------------------------------

Conversation 2:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Tell me about cats.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Cats are domestic felines.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


----------------------------------------

Conversation 3:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

How's the weather?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


----------------------------------------


## Custom Template Functions

In [14]:
def prepare_emu3_input(
    messages: List[Dict[str, str]],
    tokenizer,
    max_length: Optional[int] = None,
    include_image_tokens: bool = False,
    image_token_count: int = 4096  # 64x64 for 512x512 image
):
    """
    Prepare input for Emu3 model with optional image tokens.
    """
    # Process messages
    if include_image_tokens:
        # Inject image tokens into first user message
        messages_copy = messages.copy()
        if messages_copy and messages_copy[0]['role'] == 'user':
            # Create placeholder for image tokens
            image_placeholder = "<|img_start|>" + "<|img_token|>" * image_token_count + "<|img_end|>"
            messages_copy[0]['content'] = image_placeholder + " " + messages_copy[0]['content']
    else:
        messages_copy = messages
    
    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages_copy,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    tokens = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True if max_length else False,
        padding=False
    )
    
    return {
        'text': text,
        'input_ids': tokens['input_ids'],
        'attention_mask': tokens.get('attention_mask'),
        'num_tokens': tokens['input_ids'].shape[-1]
    }

In [15]:
# Test custom function
test_messages = [
    {"role": "user", "content": "Describe this image in detail."}
]

# Without image tokens
result_text = prepare_emu3_input(test_messages, tokenizer, include_image_tokens=False)
print("Without image tokens:")
print(f"Text: {result_text['text']}")
print(f"Tokens: {result_text['num_tokens']}")

print("\n" + "="*50 + "\n")

# With image tokens (simplified placeholder)
result_image = prepare_emu3_input(
    test_messages, 
    tokenizer, 
    include_image_tokens=True,
    image_token_count=10  # Use small number for display
)
print("With image tokens:")
print(f"Text preview (first 200 chars): {result_image['text'][:200]}...")
print(f"Tokens: {result_image['num_tokens']}")

Without image tokens:
Text: <|begin_of_text|><|start_header_id|>user<|end_header_id|>

Describe this image in detail.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


Tokens: 17


With image tokens:
Text preview (first 200 chars): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|img_start|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token|><|img_token...
Tokens: 70


## Decode Tokens Back to Text

In [16]:
# Example: tokenize and decode
sample_messages = [
    {"role": "user", "content": "Hello, how are you?"}
]

# Tokenize
tokens = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=True,
    add_generation_prompt=True
)

print(f"Token IDs: {tokens[:20]}...")  # Show first 20
print(f"Total tokens: {len(tokens)}")

# Decode back
decoded_text = tokenizer.decode(tokens, skip_special_tokens=False)
print(f"\nDecoded (with special tokens):\n{decoded_text}")

decoded_clean = tokenizer.decode(tokens, skip_special_tokens=True)
print(f"\nDecoded (without special tokens):\n{decoded_clean}")

Token IDs: [128000, 128006, 882, 128007, 271, 9906, 11, 1268, 527, 499, 30, 128009, 128006, 78191, 128007, 271]...
Total tokens: 16

Decoded (with special tokens):
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello, how are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Decoded (without special tokens):
user

Hello, how are you?assistant




## Generation Mode Tokens

In [17]:
# Load and examine generation mode tokens
import os

gen_mode_path = os.path.join(tokenizer_path, "generation_mode_token.json")
if os.path.exists(gen_mode_path):
    with open(gen_mode_path, 'r') as f:
        gen_mode_tokens = json.load(f)
    
    print("Generation mode tokens:")
    for mode, token in gen_mode_tokens.items():
        token_id = tokenizer.convert_tokens_to_ids(token) if token in tokenizer.get_vocab() else None
        print(f"  {mode}: {token} (ID: {token_id})")
else:
    print("No generation_mode_token.json found")

No generation_mode_token.json found


## Vision Token Mapping

In [18]:
# Load vision token mapping
vision_map_path = os.path.join(tokenizer_path, "vision_token_mapping.json")
if os.path.exists(vision_map_path):
    with open(vision_map_path, 'r') as f:
        vision_mapping = json.load(f)
    
    print(f"Vision token mapping loaded")
    print(f"Number of vision tokens: {len(vision_mapping)}")
    
    # Show a few examples
    print("\nSample vision token mappings (first 5):")
    for i, (k, v) in enumerate(list(vision_mapping.items())[:5]):
        print(f"  {k}: {v}")
else:
    print("No vision_token_mapping.json found")

No vision_token_mapping.json found


## Summary and Best Practices

In [19]:
# Summary of tokenizer capabilities
print("Emu3 Tokenizer Summary:")
print("="*50)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"Padding side: {tokenizer.padding_side}")
print(f"Chat template available: {hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None}")
print(f"\nSpecial tokens:")
print(f"  BOS: {tokenizer.bos_token}")
print(f"  EOS: {tokenizer.eos_token}")
print(f"  PAD: {tokenizer.pad_token}")
print(f"  Additional special tokens: {len(tokenizer.additional_special_tokens)}")

# Best practices
print("\nBest Practices:")
print("1. Always use apply_chat_template() for consistent formatting")
print("2. Set add_generation_prompt=True when preparing for generation")
print("3. Use tokenize=False to inspect the formatted text")
print("4. Handle image tokens with <|img_start|> and <|img_end|> markers")
print("5. Check token counts to ensure they fit within model limits")

Emu3 Tokenizer Summary:
Vocabulary size: 128000
Model max length: 1000000000000000019884624838656
Padding side: right
Chat template available: True

Special tokens:
  BOS: <|begin_of_text|>
  EOS: <|end_of_text|>
  PAD: None
  Additional special tokens: 32873

Best Practices:
1. Always use apply_chat_template() for consistent formatting
2. Set add_generation_prompt=True when preparing for generation
3. Use tokenize=False to inspect the formatted text
4. Handle image tokens with <|img_start|> and <|img_end|> markers
5. Check token counts to ensure they fit within model limits
